In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print(RAW)

/Users/biloshitsky_ivan/football-analytics/data/raw


In [8]:
from statsbombpy import sb
import pandas as pd

comps = sb.competitions()
comps[comps["competition_name"].str.contains("Euro")][
    ["competition_id", "season_id", "season_name"]
]

/Users/biloshitsky_ivan/football-analytics/.venv/lib/python3.12/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,season_name
73,55,282,2024
74,55,43,2020
75,35,75,1988/1989
76,53,315,2025
77,53,106,2022


In [9]:
matches = sb.matches(competition_id=55, season_id=282)
print(len(matches))  # should be 51
matches[["match_id", "match_date", "home_team", "away_team",
         "home_score", "away_score", "competition_stage"]].sort_values("match_date")

51


/Users/biloshitsky_ivan/football-analytics/.venv/lib/python3.12/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,home_team,away_team,home_score,away_score,competition_stage
50,3930158,2024-06-14,Germany,Scotland,5,1,Group Stage
1,3930159,2024-06-15,Hungary,Switzerland,1,3,Group Stage
5,3930160,2024-06-15,Spain,Croatia,3,0,Group Stage
47,3930161,2024-06-15,Italy,Albania,2,1,Group Stage
2,3930163,2024-06-16,Serbia,England,0,1,Group Stage
13,3938637,2024-06-16,Poland,Netherlands,1,2,Group Stage
18,3930162,2024-06-16,Slovenia,Denmark,1,1,Group Stage
11,3938638,2024-06-17,Romania,Ukraine,3,0,Group Stage
45,3930165,2024-06-17,Austria,France,0,1,Group Stage
46,3930164,2024-06-17,Belgium,Slovakia,0,1,Group Stage


In [10]:
final = matches[matches["competition_stage"] == "Final"].iloc[0]
ev = sb.events(match_id=final["match_id"])
ev.to_parquet("data/raw/events_final.parquet")

/Users/biloshitsky_ivan/football-analytics/.venv/lib/python3.12/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


OSError: Cannot save file into a non-existent directory: 'data/raw'

In [11]:
ev.to_parquet(RAW / "events_final.parquet")

In [12]:
ev.shape
ev["type"].value_counts()

type
Pass                 915
Ball Receipt*        876
Carry                757
Pressure             327
Ball Recovery         69
Duel                  68
Clearance             42
Block                 41
Goal Keeper           30
Dribble               25
Shot                  25
Dispossessed          21
Foul Committed        19
Foul Won              19
Miscontrol            15
Dribbled Past         11
50/50                  8
Substitution           7
Injury Stoppage        6
Interception           5
Half Start             4
Half End               4
Tactical Shift         3
Starting XI            2
Referee Ball-Drop      2
Shield                 2
Error                  1
Name: count, dtype: int64

In [15]:
[c for c in ev.columns if c.startswith("shot_")]

['shot_aerial_won',
 'shot_body_part',
 'shot_deflected',
 'shot_end_location',
 'shot_first_time',
 'shot_freeze_frame',
 'shot_key_pass_id',
 'shot_one_on_one',
 'shot_outcome',
 'shot_statsbomb_xg',
 'shot_technique',
 'shot_type']

In [16]:
passes = ev[ev["type"] == "Pass"]
accuracy = passes.groupby("team")["pass_outcome"].apply(lambda s: s.isna().mean())

In [17]:
shots = ev[ev["type"] == "Shot"].copy()

goals = shots[(shots["shot_outcome"] == "Goal") & (shots["period"] < 5)]
own_goals = ev[ev["type"] == "Own Goal For"]

score = goals.groupby("team").size().add(
    own_goals.groupby("team").size(), fill_value=0
)
print(score)
print(f"Official: {final['home_team']} {final['home_score']} – {final['away_score']} {final['away_team']}")

team
England    1.0
Spain      2.0
dtype: float64
Official: Spain 2 – 1 England


In [18]:
xg = (
    shots[shots["period"] < 5]
    .groupby("team")
    .agg(shots=("id", "count"), xg=("shot_statsbomb_xg", "sum"))
    .round(2)
)
xg["goals"] = score
xg

,shots,xg,goals
team,,,
England,9,0.73,1.0
Spain,16,1.79,2.0


In [19]:
passes = ev[ev["type"] == "Pass"]

pass_stats = passes.groupby(["team", "player"]).agg(
    passes=("id", "count"),
    accurate=("pass_outcome", lambda s: s.isna().sum()),
)
pass_stats["accuracy"] = (pass_stats["accurate"] / pass_stats["passes"]).round(3)
pass_stats.sort_values("passes", ascending=False).head(10)

passes  accurate  accuracy
team    player                                                  
Spain   Robin Aime Robert Le Normand      84        80     0.952
        Aymeric Laporte                   83        80     0.964
        Daniel Carvajal Ramos             73        63     0.863
        Fabián Ruiz Peña                  63        55     0.873
        Nicholas Williams Arthuer         51        41     0.804
        Marc Cucurella Saseta             43        42     0.977
England Jordan Pickford                   42        26     0.619
        Declan Rice                       42        32     0.762
        Kyle Walker                       42        36     0.857
Spain   Unai Simón Mendibil               40        34     0.850

In [20]:
shots[["x", "y"]] = pd.DataFrame(shots["location"].tolist(), index=shots.index)

print(shots.groupby("team")["x"].describe()[["min", "mean", "max"]])

goals = shots[(shots["shot_outcome"] == "Goal") & (shots["period"] < 5)]
goals[["minute", "team", "player", "location", "shot_end_location", "shot_statsbomb_xg"]]

          min        mean    max
team                            
England  91.0  104.333333  114.8
Spain    93.3  106.268750  115.6


,minute,team,player,location,shot_end_location,shot_statsbomb_xg
3232,46,Spain,Nicholas Williams Arthuer,"[109.9, 28.0]","[120.0, 42.4, 0.2]",0.112482
3243,72,England,Cole Palmer,"[98.4, 49.0]","[120.0, 37.1, 0.2]",0.037943
3245,85,Spain,Mikel Oyarzabal Ugarte,"[111.2, 36.3]","[120.0, 38.4, 0.2]",0.283328


In [21]:
pid = goals.sort_values(["period", "minute", "second"]).iloc[0]["possession"]

chain = ev[ev["possession"] == pid].sort_values("index")
chain[["minute", "second", "team", "player", "type", "location", "pass_recipient"]]

,minute,second,team,player,type,location,pass_recipient
493,45,35,Spain,Marc Cucurella Saseta,Pass,"[40.7, 0.1]",Aymeric Laporte
1384,45,37,Spain,Aymeric Laporte,Ball Receipt*,"[30.4, 6.1]",NaN
2192,45,37,Spain,Aymeric Laporte,Carry,"[30.4, 6.1]",NaN
494,45,38,Spain,Aymeric Laporte,Pass,"[30.4, 6.1]",Robin Aime Robert Le Normand
1385,45,39,Spain,Robin Aime Robert Le Normand,Ball Receipt*,"[27.5, 46.6]",NaN
2193,45,39,Spain,Robin Aime Robert Le Normand,Carry,"[27.5, 46.6]",NaN
495,45,41,Spain,Robin Aime Robert Le Normand,Pass,"[27.5, 52.2]",Martín Zubimendi Ibáñez
1386,45,42,Spain,Martín Zubimendi Ibáñez,Ball Receipt*,"[40.6, 51.4]",NaN
2194,45,42,Spain,Martín Zubimendi Ibáñez,Carry,"[40.6, 51.4]",NaN
496,45,49,Spain,Martín Zubimendi Ibáñez,Pass,"[48.6, 27.8]",Marc Cucurella Saseta


In [22]:
frames = sb.frames(match_id=final["match_id"])
print(frames.shape)
frames.head()

/Users/biloshitsky_ivan/football-analytics/.venv/lib/python3.12/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


(47480, 7)


,id,visible_area,match_id,teammate,actor,keeper,location
0,277afcf7-04cb-4768-b41c-2b421d657d1d,"[76.727979231533, 80.0, 43.48260274825104, 80....",3943043,True,False,False,"[48.27068629695013, 49.528033474008]"
1,277afcf7-04cb-4768-b41c-2b421d657d1d,"[76.727979231533, 80.0, 43.48260274825104, 80....",3943043,True,False,False,"[48.80936948915465, 37.18362338994975]"
2,277afcf7-04cb-4768-b41c-2b421d657d1d,"[76.727979231533, 80.0, 43.48260274825104, 80....",3943043,True,False,False,"[51.20914819207871, 24.023192465266188]"
3,277afcf7-04cb-4768-b41c-2b421d657d1d,"[76.727979231533, 80.0, 43.48260274825104, 80....",3943043,True,False,False,"[54.18371067305275, 61.931201809614144]"
4,277afcf7-04cb-4768-b41c-2b421d657d1d,"[76.727979231533, 80.0, 43.48260274825104, 80....",3943043,True,False,False,"[55.71087812383658, 50.36811216819178]"
